In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 6: Creación de datasets definitivos para modelos ML y DL.
Entrada: Carpetas generadas en el paso 5 (windows_partitioned/).
Salida:
  - Para modelos ML (RandomForest, XGBoost): versiones 2D (aplanadas) de las ventanas,
    guardadas en subcarpetas 'ml_2d' junto a los mismos archivos de partición.
  - Archivos de metadatos (JSON) que listan las estaciones disponibles y las rutas.
  - Estructura final lista para ser cargada por los scripts de entrenamiento.
"""

import os
import json
import numpy as np
from pathlib import Path

# ============================================================================
# CONFIGURACIÓN
#=============================================================================

BASE_DIR = os.path.expanduser("~/Documents/GitHub/TFGFinal/Datos_TFG_outliers")
INPUT_DIR = os.path.join(BASE_DIR, "windows_partitioned")

# Subcarpetas de entrada (generadas en código 5)
PATHS = {
    "by_station_ml": os.path.join(INPUT_DIR, "by_station", "ml"),
    "by_station_dl": os.path.join(INPUT_DIR, "by_station", "dl"),
    "global_ml": os.path.join(INPUT_DIR, "global", "ml"),
    "global_dl": os.path.join(INPUT_DIR, "global", "dl"),
}

# Parámetros de ventanas (deben coincidir con los usados antes)
WINDOW_IN = 72
WINDOW_OUT = 72

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def flatten_3d_to_2d(X_3d):
    """
    Convierte un array de forma (n_samples, window_in, n_features)
    a (n_samples, window_in * n_features).
    """
    n_samples, win_in, n_features = X_3d.shape
    return X_3d.reshape(n_samples, win_in * n_features)

def process_station_ml(ml_dir, station_name):
    """
    Para una estación en la carpeta ml (con datos 3D sin normalizar),
    genera versiones 2D aplanadas y las guarda en una subcarpeta 'ml_2d'.
    """
    station_path = os.path.join(ml_dir, station_name)
    if not os.path.isdir(station_path):
        return False

    # Cargar arrays 3D
    train_X = np.load(os.path.join(station_path, "train_X.npy"))
    val_X = np.load(os.path.join(station_path, "val_X.npy"))
    test_X = np.load(os.path.join(station_path, "test_X.npy"))
    train_y = np.load(os.path.join(station_path, "train_y.npy"))
    val_y = np.load(os.path.join(station_path, "val_y.npy"))
    test_y = np.load(os.path.join(station_path, "test_y.npy"))

    # Aplanar X
    train_X_2d = flatten_3d_to_2d(train_X)
    val_X_2d = flatten_3d_to_2d(val_X)
    test_X_2d = flatten_3d_to_2d(test_X)

    # Crear subcarpeta ml_2d
    out_dir = os.path.join(ml_dir, "ml_2d", station_name)
    os.makedirs(out_dir, exist_ok=True)

    # Guardar
    np.save(os.path.join(out_dir, "train_X.npy"), train_X_2d)
    np.save(os.path.join(out_dir, "val_X.npy"), val_X_2d)
    np.save(os.path.join(out_dir, "test_X.npy"), test_X_2d)
    np.save(os.path.join(out_dir, "train_y.npy"), train_y)
    np.save(os.path.join(out_dir, "val_y.npy"), val_y)
    np.save(os.path.join(out_dir, "test_y.npy"), test_y)

    # Copiar features.json y scalers si existen (no para ML, pero por si acaso)
    if os.path.exists(os.path.join(station_path, "features.json")):
        with open(os.path.join(station_path, "features.json"), 'r') as f:
            features = json.load(f)
        with open(os.path.join(out_dir, "features.json"), 'w') as f:
            json.dump(features, f)

    print(f"    ML 2D generado para {station_name}: {train_X_2d.shape}")
    return True

def process_global_ml(ml_dir):
    """
    Para el conjunto global en ml_dir (sin subcarpetas de estación),
    genera versiones 2D aplanadas en 'ml_2d' dentro del mismo directorio.
    """
    # Verificar que existen los archivos globales
    train_X_path = os.path.join(ml_dir, "train_X.npy")
    if not os.path.exists(train_X_path):
        print(f"    No se encuentran archivos globales en {ml_dir}")
        return

    train_X = np.load(train_X_path)
    val_X = np.load(os.path.join(ml_dir, "val_X.npy"))
    test_X = np.load(os.path.join(ml_dir, "test_X.npy"))
    train_y = np.load(os.path.join(ml_dir, "train_y.npy"))
    val_y = np.load(os.path.join(ml_dir, "val_y.npy"))
    test_y = np.load(os.path.join(ml_dir, "test_y.npy"))

    # Aplanar
    train_X_2d = flatten_3d_to_2d(train_X)
    val_X_2d = flatten_3d_to_2d(val_X)
    test_X_2d = flatten_3d_to_2d(test_X)

    # Crear subcarpeta ml_2d
    out_dir = os.path.join(ml_dir, "ml_2d")
    os.makedirs(out_dir, exist_ok=True)

    np.save(os.path.join(out_dir, "train_X.npy"), train_X_2d)
    np.save(os.path.join(out_dir, "val_X.npy"), val_X_2d)
    np.save(os.path.join(out_dir, "test_X.npy"), test_X_2d)
    np.save(os.path.join(out_dir, "train_y.npy"), train_y)
    np.save(os.path.join(out_dir, "val_y.npy"), val_y)
    np.save(os.path.join(out_dir, "test_y.npy"), test_y)

    if os.path.exists(os.path.join(ml_dir, "features.json")):
        with open(os.path.join(ml_dir, "features.json"), 'r') as f:
            features = json.load(f)
        with open(os.path.join(out_dir, "features.json"), 'w') as f:
            json.dump(features, f)

    print(f"    ML 2D global generado: {train_X_2d.shape}")

def generate_metadata():
    """
    Crea un archivo JSON con la estructura de carpetas y listado de estaciones.
    """
    metadata = {
        "by_station": {
            "ml": {
                "3d": [],
                "2d": []
            },
            "dl": []
        },
        "global": {
            "ml": {
                "3d": False,
                "2d": False
            },
            "dl": False
        }
    }

    # Por estación ML 3D
    ml_3d_dir = PATHS["by_station_ml"]
    if os.path.exists(ml_3d_dir):
        stations = [d for d in os.listdir(ml_3d_dir) if os.path.isdir(os.path.join(ml_3d_dir, d))]
        metadata["by_station"]["ml"]["3d"] = stations

    # Por estación ML 2D
    ml_2d_dir = os.path.join(ml_3d_dir, "ml_2d")
    if os.path.exists(ml_2d_dir):
        stations_2d = [d for d in os.listdir(ml_2d_dir) if os.path.isdir(os.path.join(ml_2d_dir, d))]
        metadata["by_station"]["ml"]["2d"] = stations_2d

    # Por estación DL
    dl_dir = PATHS["by_station_dl"]
    if os.path.exists(dl_dir):
        stations_dl = [d for d in os.listdir(dl_dir) if os.path.isdir(os.path.join(dl_dir, d))]
        metadata["by_station"]["dl"] = stations_dl

    # Global ML 3D
    global_ml_dir = PATHS["global_ml"]
    if os.path.exists(os.path.join(global_ml_dir, "train_X.npy")):
        metadata["global"]["ml"]["3d"] = True
    # Global ML 2D
    if os.path.exists(os.path.join(global_ml_dir, "ml_2d", "train_X.npy")):
        metadata["global"]["ml"]["2d"] = True
    # Global DL
    global_dl_dir = PATHS["global_dl"]
    if os.path.exists(os.path.join(global_dl_dir, "train_X.npy")):
        metadata["global"]["dl"] = True

    # Guardar metadata
    metadata_path = os.path.join(INPUT_DIR, "dataset_metadata.json")
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"Metadatos guardados en: {metadata_path}")

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("=" * 60)
    print("Preparación final de datasets para modelos")
    print("=" * 60)

    # 1. Procesar ML por estación (generar 2D)
    print("\n--- Generando versiones 2D para ML por estación ---")
    if os.path.exists(PATHS["by_station_ml"]):
        ml_dir = PATHS["by_station_ml"]
        stations = [d for d in os.listdir(ml_dir) if os.path.isdir(os.path.join(ml_dir, d)) and d != "ml_2d"]
        for station in stations:
            process_station_ml(ml_dir, station)
    else:
        print("  No existe la carpeta by_station/ml")

    # 2. Procesar ML global
    print("\n--- Generando versiones 2D para ML global ---")
    if os.path.exists(PATHS["global_ml"]):
        process_global_ml(PATHS["global_ml"])
    else:
        print("  No existe la carpeta global/ml")

    # 3. Generar metadatos
    print("\n--- Generando metadatos ---")
    generate_metadata()

    print("\nProceso completado. Los datasets están listos en:")
    print(f"  {INPUT_DIR}")
    print("\nEstructura generada:")
    print("  by_station/ml/            -> datos 3D sin normalizar (por estación)")
    print("  by_station/ml/ml_2d/      -> datos 2D aplanados (para árboles)")
    print("  by_station/dl/            -> datos 3D normalizados (para RNN/LSTM)")
    print("  global/ml/                 -> datos 3D globales sin normalizar")
    print("  global/ml/ml_2d/           -> datos 2D globales aplanados")
    print("  global/dl/                 -> datos 3D globales normalizados")
    print("  dataset_metadata.json      -> resumen de lo disponible")
